# Library

In [1]:
import sys
sys.path.append("/project/assistant")

%load_ext autoreload
%autoreload 2

import os

# huggingface downloads: quiet text logs, no widget progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"

import frontmatter
from elasticsearch import Elasticsearch

from cleaner import Document, NoteCleaner
from embeddings import SentenceTransformerEmbeddings
from ingest import VaultLoader, SlidingWindowSplitter, ElasticsearchIndexer

# Data Cleaner

The package follows the LangChain document model without depending on LangChain itself. Every piece of text is represented as a `Document`, using the structure commonly adopted across the LLM ecosystem: `page_content` stores the text, while `metadata` stores the associated information.

The metadata schema follows the LangChain `ObsidianLoader` convention (`source`, `last_modified`) and extends it with the additional information captured by this project: `folder`, `tags`, weighted note references (`graph_edges`), and external URLs.

The cleaner below is imported directly from the package (`assistant/cleaner.py`) and tested against `data/cleaner_stress_test.md`, a note designed to stress every cleaning rule. It contains all supported link and embed variations, placed inside lists, blockquotes, tables, and code blocks, where parsing and cleaning are most likely to fail.

Running the cleaner against this single note makes it possible to inspect every supported transformation, together with the one documented limitation, in a single output.


In [2]:
cleaner = NoteCleaner()

raw = frontmatter.load("/project/data/cleaner_stress_test.md")
cleaned, graph_edges, external_links = cleaner.clean(raw.content)

print(cleaned)
print("=" * 60)
print("graph edges (weighted):", dict(graph_edges))
print("external links:", external_links)

# Cleaner torture test

This note exists to break the NoteCleaner. Every link and embed variation

lives here, mixed into realistic prose, lists, quotes, tables and code.

## Plain references

The basics: see 14-agent-evaluation for evaluation and

the tools note when you need function calling.

A link with extension 05-search should behave like one without.

Path qualified: 03-rag points across folders.

Versioned names are cruel: Restart Dataset - 1.6.0 has dots that are not extensions.

Names with spaces exist too: My Daily Notes Setup.

## Anchors and blocks

Heading anchor: 05-search and nested 05-search.

Block reference: 2026-01-01 and a readable one quotes.

Same note heading: #Plain references and same note block: #^localblock.

## Attachments referenced as links

The figure Figure 1.png and the sheet budget.pdf are wikilinked, not embedded.

## Embeds

Pasted screenshot:

Sized both ways:  and

Full note transclusion: important-note

Section transclusion: 14-agent-evaluation


# Data Ingestion

The loader walks through the vault and converts note files into `Document` objects, following the LangChain loader interface: `lazy_load()` yields documents one at a time, while `load()` returns the complete collection.

This stage is responsible for file discovery and frontmatter parsing. Text cleaning is delegated to the injected cleaner, while chunking is handled separately by the splitter described below.


In [3]:
loader = VaultLoader(os.getenv("VAULT_PATH", "/vault"), cleaner)

docs = loader.load()
print(len(docs), "documents")
docs[10].show()

66 documents
Title: 06-building-prompt   Folder: [llm-zoomcamp-2026/01-agentic-rag]
source:          llm-zoomcamp-2026/01-agentic-rag/06-building-prompt.md
last modified:   2026-07-17T19:50:30
tags:            (none)
graph edges:     
   05-search : 1
   07-llm : 1
external links:  
    https://www.youtube.com/watch?v=DV4e2n-dIv0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv

content: 3510 chars, first 400:
------------------------------------------------------------------------

# Building the Prompt

Video: Watch this lesson

The LLM doesn't see our documents unless we pass them in. So we need
to build a prompt that includes the user's question and the search
results.

When we build AI systems, we usually split the prompt into two parts:

- Instructions (also called the system prompt): this tells the LLM how
  to behave. It never changes, so it's the same for every reque
------------------------------------------------------------------------


Ingestion moves the notes from the vault into the search engine through three components that follow the LangChain pipeline: the loader converts files into `Document` objects, the splitter breaks those documents into overlapping chunks, and the indexer writes the chunks to Elasticsearch together with their embeddings.

The loader was already exercised in the cleaner section above. The following sections focus on the splitter and the indexer.


## **Chunking (Sliding Window)**

In [4]:
splitter = SlidingWindowSplitter(chunk_size=2000, chunk_overlap=1000)

chunks = splitter.split_documents(docs)
print(len(docs), "documents ->", len(chunks), "chunks")
chunks[1].metadata

66 documents -> 340 chunks


{'source': 'concepts/evaluation.md',
 'title': 'evaluation',
 'folder': 'concepts',
 'tags': ['concepts', '04-evaluation'],
 'graph_edges': {},
 'external_links': [],
 'last_modified': '2026-08-09T17:15:18.609023+00:00',
 'start': 1000}

## **Embeddings:** Sentence Transformers

In [5]:
model_name = os.getenv("EMBED_MODEL", "all-MiniLM-L6-v2")
query_prefix = ("Represent this sentence for searching relevant passages: "
                if "bge" in model_name else "")
embeddings = SentenceTransformerEmbeddings(model_name, query_prefix)

vector = embeddings.embed_query("what is retrieval augmented generation?")
print(model_name, "| dims:", len(vector))

all-MiniLM-L6-v2 | dims: 384


## **Indexing**: Elasticsearch

In [6]:
ES_INDEX = os.getenv("ES_INDEX", "obsidian_notes")
es = Elasticsearch(os.getenv("ELASTIC_HOST", "http://elasticsearch:9200"))
indexer = ElasticsearchIndexer(es, embeddings, index=ES_INDEX, add_context_header=True)

indexer.create_index(recreate=True)
indexer.index_documents(chunks)

es.indices.refresh(index=indexer.index)
es.count(index=indexer.index)

ObjectApiResponse({'count': 340, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})

In [7]:
first = es.search(index=indexer.index, size=1, query={"match_all": {}})["hits"]["hits"][0]
print("id:", first["_id"])

src = first["_source"]
for k, v in src.items():
    if k == "embedding":
        print(f"embedding: {len(v)} dims")
    elif k == "content":
        print(f"content:\n{v[:200]}...")
    else:
        print(f"{k}: {v}")

id: concepts/evaluation.md::0
content:
[concepts / evaluation]
# evaluation

Raw reference: the explanatory markdown of `04-evaluation/notebooks/evaluation.ipynb` (module 4 of the
course), transcribed in original order. The project notes c...
title: evaluation
path: concepts/evaluation.md
folder: concepts
tags: ['concepts', '04-evaluation']
graph_edges: []
external_links: []
start: 0
modified_at: 2026-08-09T17:15:18.609023+00:00
embedding: 384 dims
